# Sign Language Detection with Faster R-CNN

This notebook implements a complete ML pipeline for detecting sign language alphabets (A-Z) using a custom Faster R-CNN model built with PyTorch. The system detects and localizes hand gestures representing the 26 English alphabet letters.

## Pipeline Overview:
1. **Custom Dataset Class** - Loading images and PASCAL VOC XML annotations
2. **Preprocessing & Augmentation** - Using torchvision.transforms.v2
3. **Model Definition** - Faster R-CNN with modified head for 27 classes
4. **Training Loop** - With optimizer, scheduler, and checkpointing
5. **Evaluation** - Using Mean Average Precision (mAP)
6. **Inference & Visualization** - Drawing bounding boxes with predictions
7. **Feature Map Visualization** - Using forward hooks to inspect model internals

## 1. Import Required Libraries

In [ ]:
# Core libraries
import os
import glob
import random
import numpy as np
from PIL import Image
from typing import Dict, List, Tuple, Optional
import xml.etree.ElementTree as ET

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# TorchVision for model and transforms
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    fasterrcnn_mobilenet_v3_large_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torchvision.transforms.v2 as T
from torchvision import tv_tensors
from torchvision.tv_tensors import BoundingBoxes, BoundingBoxFormat

# Evaluation metrics
from torchmetrics.detection import MeanAveragePrecision

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Class mapping: 0 = background, 1-26 = A-Z
ALPHABET = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
CLASS_TO_IDX = {letter: idx + 1 for idx, letter in enumerate(ALPHABET)}  # 1-indexed (0 is background)
IDX_TO_CLASS = {idx + 1: letter for idx, letter in enumerate(ALPHABET)}
NUM_CLASSES = 27  # 26 letters + 1 background

print(f"Number of classes: {NUM_CLASSES}")
print(f"Class mapping: {CLASS_TO_IDX}")

## 2. Custom Dataset Class for Sign Language Detection

This dataset class handles loading images and parsing PASCAL VOC XML annotations. 
The XML format stores bounding box coordinates and class labels for each sign language gesture.

**Expected directory structure (Roboflow format):**
```
data/
├── train/
│   ├── img_001.jpg
│   ├── img_001.xml
│   ├── img_002.jpg
│   ├── img_002.xml
│   └── ...
├── valid/
│   ├── img_003.jpg
│   ├── img_003.xml

│   └── ...*Note: Images and annotations are co-located in the same directory (Roboflow PASCAL VOC export format).*

└── test/

    ├── img_004.jpg```

    ├── img_004.xml    └── ...

In [ ]:
def parse_voc_xml(xml_path: str) -> Tuple[List[List[int]], List[int]]:
    """
    Parse PASCAL VOC format XML annotation file.
    
    Args:
        xml_path: Path to the XML annotation file
        
    Returns:
        boxes: List of bounding boxes in [x_min, y_min, x_max, y_max] format
        labels: List of class indices (1-26 for A-Z)
    
    Expected XML structure:
    <annotation>
        <object>
            <name>A</name>  <!-- Letter class -->
            <bndbox>
                <xmin>100</xmin>
                <ymin>50</ymin>
                <xmax>200</xmax>
                <ymax>150</ymax>
            </bndbox>
        </object>
        ...
    </annotation>
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    
    boxes = []
    labels = []
    
    for obj in root.findall('object'):
        # Get class name (letter A-Z)
        name = obj.find('name').text.upper()
        
        # Skip if not a valid alphabet letter
        if name not in CLASS_TO_IDX:
            continue
            
        label = CLASS_TO_IDX[name]
        
        # Parse bounding box coordinates
        bbox = obj.find('bndbox')
        x_min = int(float(bbox.find('xmin').text))
        y_min = int(float(bbox.find('ymin').text))
        x_max = int(float(bbox.find('xmax').text))
        y_max = int(float(bbox.find('ymax').text))
        
        # Validate box coordinates
        if x_max > x_min and y_max > y_min:
            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(label)
    
    return boxes, labels


class SignLanguageDataset(Dataset):
    """
    Custom PyTorch Dataset for Sign Language Detection.
    
    Loads images and parses PASCAL VOC XML format annotations.
    Returns images as tensors and targets as dictionaries containing
    bounding boxes, labels, and image IDs.
    
    Supports Roboflow export format where images and annotations
    are co-located in the same directory.
    """
    
    def __init__(
        self, 
        data_dir: str, 
        transforms: Optional[T.Compose] = None
    ):
        """
        Args:
            data_dir: Directory containing both images AND XML annotation files
                     (Roboflow co-located format: train/, valid/, or test/ folder)
            transforms: Optional torchvision transforms to apply
        """
        self.data_dir = data_dir
        self.transforms = transforms
        
        # Get list of image files (support common formats)
        self.image_files = sorted(
            glob.glob(os.path.join(data_dir, "*.jpg")) +
            glob.glob(os.path.join(data_dir, "*.jpeg")) +
            glob.glob(os.path.join(data_dir, "*.png"))
        )
        
        # Filter to only include images with corresponding annotations (in same directory)
        self.valid_samples = []
        for img_path in self.image_files:
            # Get corresponding XML annotation path (same directory, same base name)
            img_name = os.path.splitext(os.path.basename(img_path))[0]
            xml_path = os.path.join(data_dir, f"{img_name}.xml")
            
            if os.path.exists(xml_path):
                self.valid_samples.append((img_path, xml_path))
        
        print(f"Found {len(self.valid_samples)} valid image-annotation pairs in {data_dir}")
    
    def __len__(self) -> int:
        return len(self.valid_samples)
    
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict]:
        img_path, xml_path = self.valid_samples[idx]
        
        # Load image
        image = Image.open(img_path).convert("RGB")
        img_width, img_height = image.size
        
        # Parse XML annotations
        boxes, labels = parse_voc_xml(xml_path)
        
        # Handle empty annotations (no valid objects in image)
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Create target dictionary (Faster R-CNN format)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros((0,)),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }
        
        # Convert image to tensor
        image = T.functional.to_image(image)
        image = T.functional.to_dtype(image, dtype=torch.float32, scale=True)
        
        # Apply transforms using tv_tensors for proper bounding box handling
        if self.transforms is not None:
            # Wrap boxes as tv_tensors.BoundingBoxes for proper transformation handling
            boxes_tv = BoundingBoxes(
                target["boxes"], 
                format=BoundingBoxFormat.XYXY, 
                canvas_size=(img_height, img_width)
            )
            
            # Apply transforms
            image, boxes_tv, labels = self.transforms(image, boxes_tv, target["labels"])
            
            # Update target with transformed boxes
            target["boxes"] = boxes_tv.data if hasattr(boxes_tv, 'data') else boxes_tv
            target["labels"] = labels
            
            # Recalculate area after transformation
            if len(target["boxes"]) > 0:
                target["area"] = (target["boxes"][:, 3] - target["boxes"][:, 1]) * \
                                 (target["boxes"][:, 2] - target["boxes"][:, 0])
        
        return image, target


# YOLO format parser (alternative annotation format)
def parse_yolo_txt(txt_path: str, img_width: int, img_height: int) -> Tuple[List[List[int]], List[int]]:
    """
    Parse YOLO format TXT annotation file.
    
    YOLO format: class_id center_x center_y width height (normalized 0-1)
    
    Args:
        txt_path: Path to the TXT annotation file
        img_width: Image width in pixels
        img_height: Image height in pixels
        
    Returns:
        boxes: List of bounding boxes in [x_min, y_min, x_max, y_max] format
        labels: List of class indices (1-indexed, adding 1 since YOLO uses 0-indexed)
    """
    boxes = []
    labels = []
    
    with open(txt_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
                
            class_id = int(parts[0]) + 1  # Convert from YOLO 0-indexed to our 1-indexed
            
            # Skip invalid class IDs
            if class_id < 1 or class_id > 26:
                continue
                
            # Parse normalized coordinates
            center_x = float(parts[1]) * img_width
            center_y = float(parts[2]) * img_height
            width = float(parts[3]) * img_width
            height = float(parts[4]) * img_height
            
            # Convert to corner format
            x_min = int(center_x - width / 2)
            y_min = int(center_y - height / 2)
            x_max = int(center_x + width / 2)
            y_max = int(center_y + height / 2)
            
            # Clamp coordinates to image bounds
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_width, x_max)
            y_max = min(img_height, y_max)
            
            if x_max > x_min and y_max > y_min:
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(class_id)
    
    return boxes, labels

print("Dataset class defined successfully!")

## 3. Preprocessing and Data Augmentation Pipeline

Using `torchvision.transforms.v2` for modern, composable transforms that properly handle bounding boxes along with images.

In [ ]:
def get_train_transforms() -> T.Compose:
    """
    Training transforms with data augmentation.
    
    Augmentations applied:
    - RandomHorizontalFlip: Flips image and bounding boxes horizontally (50% chance)
    - ColorJitter: Random brightness, contrast, saturation, and hue variations
    - RandomPhotometricDistort: Advanced photometric augmentation
    - SanitizeBoundingBoxes: Removes degenerate/invalid boxes after transforms
    
    Note: We don't apply RandomResize as Faster R-CNN handles multi-scale internally.
    """
    return T.Compose([
        # Geometric augmentations
        T.RandomHorizontalFlip(p=0.5),
        
        # Photometric augmentations
        T.ColorJitter(
            brightness=0.3,
            contrast=0.3,
            saturation=0.3,
            hue=0.1
        ),
        T.RandomPhotometricDistort(p=0.5),
        
        # Optional: Random scaling (Faster R-CNN handles this, but can help with diversity)
        # T.RandomShortestSize(
        #     min_size=(480, 512, 544, 576, 608, 640),
        #     max_size=1024
        # ),
        
        # Clean up any invalid bounding boxes that may have been produced
        T.SanitizeBoundingBoxes(min_size=1.0),
    ])


def get_val_transforms() -> T.Compose:
    """
    Validation/Test transforms (minimal preprocessing, no augmentation).
    
    Only includes essential transforms for inference consistency.
    """
    return T.Compose([
        # No augmentation for validation - just ensure clean boxes
        T.SanitizeBoundingBoxes(min_size=1.0),
    ])


def collate_fn(batch: List[Tuple[torch.Tensor, Dict]]) -> Tuple[List[torch.Tensor], List[Dict]]:
    """
    Custom collate function for DataLoader.
    
    Faster R-CNN expects a list of images and a list of target dictionaries,
    not batched tensors, because images may have different numbers of objects.
    
    Args:
        batch: List of (image, target) tuples
        
    Returns:
        Tuple of (list of images, list of targets)
    """
    images = []
    targets = []
    
    for image, target in batch:
        images.append(image)
        targets.append(target)
    
    return images, targets


print("Transforms and collate function defined!")
print("\nTraining transforms:")
print(get_train_transforms())
print("\nValidation transforms:")
print(get_val_transforms())

## 4. Data Loaders Setup

Split dataset into training and validation sets, then create DataLoader instances with the custom collate function.

In [ ]:
# ==========================================
# CONFIGURATION - Roboflow Dataset Paths
# ==========================================
# Roboflow exports data with train/valid/test splits already done
# Images and annotations are co-located in each split folder

DATA_ROOT = "./data"                     # Root data directory
TRAIN_DIR = os.path.join(DATA_ROOT, "train")   # Training data
VALID_DIR = os.path.join(DATA_ROOT, "valid")   # Validation data
TEST_DIR = os.path.join(DATA_ROOT, "test")     # Test data

BATCH_SIZE = 4
NUM_WORKERS = 2  # Adjust based on your system (use 0 on Windows if issues)

# ==========================================
# Create datasets with transforms
# ==========================================

def create_data_loaders(
    train_dir: str,
    valid_dir: str,
    batch_size: int = 4,
    num_workers: int = 2
) -> Tuple[DataLoader, DataLoader]:
    """
    Create training and validation data loaders using Roboflow pre-split directories.
    
    Args:
        train_dir: Path to training data directory (images + annotations)
        valid_dir: Path to validation data directory (images + annotations)
        batch_size: Batch size for DataLoaders
        num_workers: Number of worker processes
        
    Returns:
        Tuple of (train_loader, val_loader)
    """
    # Create training dataset with augmentation transforms
    train_dataset = SignLanguageDataset(
        data_dir=train_dir,
        transforms=get_train_transforms()
    )
    
    # Create validation dataset with minimal transforms (no augmentation)
    val_dataset = SignLanguageDataset(
        data_dir=valid_dir,
        transforms=get_val_transforms()
    )
    
    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"\nDataset Statistics:")
    print(f"  Training samples: {len(train_dataset)}")
    print(f"  Validation samples: {len(val_dataset)}")
    print(f"  Batch size: {batch_size}")
    print(f"  Training batches: {len(train_loader)}")
    print(f"  Validation batches: {len(val_loader)}")
    
    return train_loader, val_loader


def create_test_loader(
    test_dir: str,
    batch_size: int = 4,
    num_workers: int = 2
) -> DataLoader:
    """
    Create test data loader for final evaluation.
    
    Args:
        test_dir: Path to test data directory (images + annotations)
        batch_size: Batch size for DataLoader
        num_workers: Number of worker processes
        
    Returns:
        test_loader DataLoader
    """
    test_dataset = SignLanguageDataset(
        data_dir=test_dir,
        transforms=get_val_transforms()  # No augmentation for test
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_fn,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"\nTest Dataset: {len(test_dataset)} samples, {len(test_loader)} batches")
    
    return test_loader


# Uncomment when you have actual data:
# train_loader, val_loader = create_data_loaders(
#     images_dir=IMAGES_DIR,
#     annotations_dir=ANNOTATIONS_DIR,
#     batch_size=BATCH_SIZE,
#     num_workers=NUM_WORKERS,
#     train_val_split=TRAIN_VAL_SPLIT
# )

print("Data loader setup function ready!")

## 5. Faster R-CNN Model Definition and Head Modification

Load a pre-trained Faster R-CNN model and modify the classification head for our 27-class problem (26 letters + 1 background).

In [ ]:
def create_faster_rcnn_model(
    num_classes: int = 27,
    backbone: str = "resnet50",
    pretrained: bool = True,
    trainable_backbone_layers: int = 3
) -> nn.Module:
    """
    Create a Faster R-CNN model with a custom number of classes.
    
    Args:
        num_classes: Number of classes (including background)
        backbone: Either "resnet50" or "mobilenet_v3"
        pretrained: Whether to use pretrained weights
        trainable_backbone_layers: Number of backbone layers to train (0-5)
        
    Returns:
        Modified Faster R-CNN model
        
    Architecture Overview:
    ┌─────────────────────────────────────────────────────────────┐
    │                    Faster R-CNN Architecture                 │
    ├─────────────────────────────────────────────────────────────┤
    │  Input Image                                                 │
    │       ↓                                                      │
    │  ┌─────────────┐                                            │
    │  │  Backbone   │  (ResNet50/MobileNetV3 + FPN)              │
    │  │  + FPN      │  Extracts multi-scale feature maps         │
    │  └─────────────┘                                            │
    │       ↓                                                      │
    │  ┌─────────────┐                                            │
    │  │     RPN     │  Region Proposal Network                   │
    │  │             │  Generates object proposals                │
    │  └─────────────┘                                            │
    │       ↓                                                      │
    │  ┌─────────────┐                                            │
    │  │  ROI Heads  │  Region of Interest processing             │
    │  │             │  - ROI Pooling                             │
    │  │             │  - Box Predictor (classification + bbox)   │
    │  └─────────────┘                                            │
    │       ↓                                                      │
    │  Output: boxes, labels, scores                               │
    └─────────────────────────────────────────────────────────────┘
    """
    
    if backbone == "resnet50":
        # Load pre-trained Faster R-CNN with ResNet-50 backbone
        if pretrained:
            weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
            model = fasterrcnn_resnet50_fpn(
                weights=weights,
                trainable_backbone_layers=trainable_backbone_layers
            )
        else:
            model = fasterrcnn_resnet50_fpn(
                weights=None,
                trainable_backbone_layers=trainable_backbone_layers
            )
        print(f"Loaded Faster R-CNN with ResNet-50 FPN backbone")
        
    elif backbone == "mobilenet_v3":
        # Load pre-trained Faster R-CNN with MobileNetV3 backbone (lighter, faster)
        if pretrained:
            weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
            model = fasterrcnn_mobilenet_v3_large_fpn(
                weights=weights,
                trainable_backbone_layers=trainable_backbone_layers
            )
        else:
            model = fasterrcnn_mobilenet_v3_large_fpn(
                weights=None,
                trainable_backbone_layers=trainable_backbone_layers
            )
        print(f"Loaded Faster R-CNN with MobileNetV3-Large FPN backbone")
        
    else:
        raise ValueError(f"Unknown backbone: {backbone}. Choose 'resnet50' or 'mobilenet_v3'")
    
    # ==========================================
    # Modify the box predictor head for our classes
    # ==========================================
    # The default model is trained on COCO with 91 classes
    # We need to replace the classification head with our 27 classes
    
    # Get the number of input features from the existing box predictor
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    print(f"Box predictor input features: {in_features}")
    
    # Replace the box predictor with a new one for our number of classes
    # FastRCNNPredictor handles both classification and bounding box regression
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    print(f"Replaced box predictor for {num_classes} classes")
    
    return model


# Create the model
model = create_faster_rcnn_model(
    num_classes=NUM_CLASSES,
    backbone="resnet50",  # Can change to "mobilenet_v3" for faster inference
    pretrained=True,
    trainable_backbone_layers=3  # Fine-tune last 3 layers of backbone
)

# Move model to device
model = model.to(DEVICE)

# Print model summary
print(f"\nModel moved to: {DEVICE}")
print(f"\nModel structure (abbreviated):")
print(f"  - Backbone: {type(model.backbone).__name__}")
print(f"  - RPN: {type(model.rpn).__name__}")
print(f"  - ROI Heads: {type(model.roi_heads).__name__}")
print(f"  - Box Predictor: {type(model.roi_heads.box_predictor).__name__}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 6. Training Loop with Optimizer and Scheduler

Complete training loop with:
- AdamW optimizer with weight decay for regularization
- StepLR learning rate scheduler for gradual learning rate decay
- Loss tracking and model checkpointing
- Progress logging per epoch

In [ ]:
# ==========================================
# Training Configuration
# ==========================================
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0005
NUM_EPOCHS = 25
LR_STEP_SIZE = 7
LR_GAMMA = 0.1
CHECKPOINT_DIR = "./checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def train_one_epoch(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    data_loader: DataLoader,
    device: torch.device,
    epoch: int
) -> float:
    """
    Train the model for one epoch.
    
    Faster R-CNN returns a dict of losses during training:
    - loss_classifier: Classification loss for ROI heads
    - loss_box_reg: Bounding box regression loss for ROI heads  
    - loss_objectness: RPN objectness loss
    - loss_rpn_box_reg: RPN bounding box regression loss
    
    Args:
        model: The Faster R-CNN model
        optimizer: The optimizer
        data_loader: Training data loader
        device: Device to train on
        epoch: Current epoch number
        
    Returns:
        Average loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = len(data_loader)
    
    for batch_idx, (images, targets) in enumerate(data_loader):
        # Move data to device
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass - model returns losses when targets are provided
        loss_dict = model(images, targets)
        
        # Sum all losses
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        losses.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        
        # Update weights
        optimizer.step()
        
        # Accumulate loss
        total_loss += losses.item()
        
        # Print progress every 10 batches
        if (batch_idx + 1) % 10 == 0:
            avg_loss = total_loss / (batch_idx + 1)
            print(f"  Epoch {epoch+1}, Batch {batch_idx+1}/{num_batches}, "
                  f"Loss: {losses.item():.4f}, Avg Loss: {avg_loss:.4f}")
    
    return total_loss / num_batches


def validate(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device
) -> float:
    """
    Validate the model and compute average loss.
    
    Note: For validation loss, we still need targets to compute losses.
    For mAP evaluation, use the evaluate_map function separately.
    
    Args:
        model: The Faster R-CNN model
        data_loader: Validation data loader
        device: Device to evaluate on
        
    Returns:
        Average validation loss
    """
    model.train()  # Keep in train mode to get losses (eval mode returns predictions)
    total_loss = 0.0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Get losses
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()
    
    return total_loss / len(data_loader)


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    num_epochs: int,
    learning_rate: float,
    weight_decay: float,
    lr_step_size: int,
    lr_gamma: float,
    device: torch.device,
    checkpoint_dir: str
) -> Dict[str, List[float]]:
    """
    Complete training loop with checkpointing.
    
    Args:
        model: The Faster R-CNN model
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Number of epochs to train
        learning_rate: Initial learning rate
        weight_decay: L2 regularization strength
        lr_step_size: Epochs between LR reductions
        lr_gamma: LR reduction factor
        device: Device to train on
        checkpoint_dir: Directory to save checkpoints
        
    Returns:
        Dictionary with training history
    """
    # Optimizer - AdamW with decoupled weight decay
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    
    # Alternative: SGD with momentum
    # optimizer = torch.optim.SGD(
    #     model.parameters(),
    #     lr=learning_rate,
    #     momentum=0.9,
    #     weight_decay=weight_decay
    # )
    
    # Learning rate scheduler - reduce LR by gamma every step_size epochs
    lr_scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=lr_step_size,
        gamma=lr_gamma
    )
    
    # Alternative: Cosine annealing
    # lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer,
    #     T_max=num_epochs
    # )
    
    # Training history
    history = {
        "train_loss": [],
        "val_loss": [],
        "learning_rate": []
    }
    
    best_val_loss = float('inf')
    
    print(f"Starting training for {num_epochs} epochs...")
    print(f"Optimizer: {type(optimizer).__name__}")
    print(f"Scheduler: {type(lr_scheduler).__name__}")
    print("-" * 50)
    
    for epoch in range(num_epochs):
        current_lr = optimizer.param_groups[0]['lr']
        print(f"\nEpoch {epoch+1}/{num_epochs} (LR: {current_lr:.6f})")
        
        # Train
        train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
        history["train_loss"].append(train_loss)
        
        # Validate
        val_loss = validate(model, val_loader, device)
        history["val_loss"].append(val_loss)
        
        # Track learning rate
        history["learning_rate"].append(current_lr)
        
        # Step scheduler
        lr_scheduler.step()
        
        print(f"Epoch {epoch+1} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
        
        # Save checkpoint if best validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            checkpoint_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
            }, checkpoint_path)
            print(f"  ✓ Saved best model (val_loss: {val_loss:.4f})")
        
        # Save periodic checkpoint
        if (epoch + 1) % 5 == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f"checkpoint_epoch_{epoch+1}.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
            }, checkpoint_path)
            print(f"  ✓ Saved checkpoint at epoch {epoch+1}")
    
    print("\nTraining complete!")
    return history


# ==========================================
# Run training (uncomment when data is ready)
# ==========================================

# history = train_model(
#     model=model,
#     train_loader=train_loader,
#     val_loader=val_loader,
#     num_epochs=NUM_EPOCHS,
#     learning_rate=LEARNING_RATE,
#     weight_decay=WEIGHT_DECAY,
#     lr_step_size=LR_STEP_SIZE,
#     lr_gamma=LR_GAMMA,
#     device=DEVICE,
#     checkpoint_dir=CHECKPOINT_DIR
# )

print("Training functions defined!")

## 7. Model Evaluation with Mean Average Precision (mAP)

Evaluate the trained model using `torchmetrics.detection.MeanAveragePrecision` which computes:
- **mAP**: Mean Average Precision averaged over IoU thresholds 0.50:0.95
- **mAP_50**: Mean AP at IoU threshold 0.50
- **mAP_75**: Mean AP at IoU threshold 0.75
- **Per-class AP**: Average Precision for each class

In [ ]:
@torch.no_grad()
def evaluate_map(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
    iou_type: str = "bbox"
) -> Dict[str, float]:
    """
    Evaluate model using Mean Average Precision (mAP).
    
    Uses torchmetrics.detection.MeanAveragePrecision which follows
    COCO evaluation protocol.
    
    Args:
        model: Trained Faster R-CNN model
        data_loader: Validation/Test data loader
        device: Device to evaluate on
        iou_type: Type of IoU computation ("bbox" for bounding boxes)
        
    Returns:
        Dictionary containing mAP metrics
    """
    model.eval()
    
    # Initialize mAP metric
    # class_metrics=True enables per-class AP computation
    map_metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type=iou_type,
        class_metrics=True
    )
    
    print("Running inference for mAP evaluation...")
    
    for batch_idx, (images, targets) in enumerate(data_loader):
        # Move images to device
        images = [img.to(device) for img in images]
        
        # Get predictions (model in eval mode returns predictions)
        predictions = model(images)
        
        # Format predictions for torchmetrics
        # Each prediction dict needs: boxes, scores, labels
        preds = []
        for pred in predictions:
            preds.append({
                "boxes": pred["boxes"].cpu(),
                "scores": pred["scores"].cpu(),
                "labels": pred["labels"].cpu()
            })
        
        # Format targets for torchmetrics
        # Each target dict needs: boxes, labels
        tgts = []
        for target in targets:
            tgts.append({
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu()
            })
        
        # Update metric with batch
        map_metric.update(preds, tgts)
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed batch {batch_idx + 1}/{len(data_loader)}")
    
    # Compute final metrics
    print("Computing mAP metrics...")
    results = map_metric.compute()
    
    # Extract key metrics
    metrics = {
        "mAP": results["map"].item(),
        "mAP_50": results["map_50"].item(),
        "mAP_75": results["map_75"].item(),
        "mAP_small": results["map_small"].item(),
        "mAP_medium": results["map_medium"].item(),
        "mAP_large": results["map_large"].item(),
    }
    
    # Print results
    print("\n" + "=" * 50)
    print("Mean Average Precision (mAP) Results")
    print("=" * 50)
    print(f"  mAP (IoU=0.50:0.95): {metrics['mAP']:.4f}")
    print(f"  mAP_50 (IoU=0.50):   {metrics['mAP_50']:.4f}")
    print(f"  mAP_75 (IoU=0.75):   {metrics['mAP_75']:.4f}")
    print(f"  mAP_small:           {metrics['mAP_small']:.4f}")
    print(f"  mAP_medium:          {metrics['mAP_medium']:.4f}")
    print(f"  mAP_large:           {metrics['mAP_large']:.4f}")
    print("=" * 50)
    
    # Per-class AP if available
    if "map_per_class" in results and results["map_per_class"].numel() > 0:
        print("\nPer-Class Average Precision:")
        print("-" * 40)
        map_per_class = results["map_per_class"]
        classes_present = results.get("classes", torch.arange(1, len(map_per_class) + 1))
        
        for i, (cls_idx, ap) in enumerate(zip(classes_present, map_per_class)):
            cls_idx = int(cls_idx.item()) if torch.is_tensor(cls_idx) else cls_idx
            if cls_idx in IDX_TO_CLASS:
                class_name = IDX_TO_CLASS[cls_idx]
            else:
                class_name = f"Class {cls_idx}"
            ap_value = ap.item() if torch.is_tensor(ap) else ap
            if ap_value >= 0:  # -1 indicates no samples for this class
                print(f"  {class_name}: {ap_value:.4f}")
        print("-" * 40)
    
    return metrics


def plot_training_history(history: Dict[str, List[float]]):
    """
    Plot training and validation loss curves.
    
    Args:
        history: Dictionary with training history
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history["train_loss"]) + 1)
    
    # Loss plot
    axes[0].plot(epochs, history["train_loss"], 'b-', label="Train Loss", linewidth=2)
    axes[0].plot(epochs, history["val_loss"], 'r-', label="Val Loss", linewidth=2)
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training and Validation Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Learning rate plot
    axes[1].plot(epochs, history["learning_rate"], 'g-', linewidth=2)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Learning Rate")
    axes[1].set_title("Learning Rate Schedule")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Training history saved to 'training_history.png'")


# ==========================================
# Run evaluation (uncomment when model is trained)
# ==========================================

# # Load best model
# checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pth"))
# model.load_state_dict(checkpoint['model_state_dict'])
# print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

# # Evaluate
# metrics = evaluate_map(model, val_loader, DEVICE)

# # Plot training history
# plot_training_history(history)

print("Evaluation functions defined!")

## 8. Inference Function with Bounding Box Visualization

Run inference on test images and visualize predictions with bounding boxes, class labels, and confidence scores.

In [ ]:
# Color palette for visualization (one color per class)
COLORS = plt.cm.tab20(np.linspace(0, 1, 27))


@torch.no_grad()
def predict_single_image(
    model: nn.Module,
    image_path: str,
    device: torch.device,
    confidence_threshold: float = 0.5
) -> Tuple[np.ndarray, List[Dict]]:
    """
    Run inference on a single image.
    
    Args:
        model: Trained Faster R-CNN model
        image_path: Path to the image
        device: Device to run inference on
        confidence_threshold: Minimum confidence to keep predictions
        
    Returns:
        Tuple of (original image as numpy array, list of filtered predictions)
    """
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert("RGB")
    original_image = np.array(image)
    
    # Convert to tensor
    image_tensor = T.functional.to_image(image)
    image_tensor = T.functional.to_dtype(image_tensor, dtype=torch.float32, scale=True)
    image_tensor = image_tensor.to(device)
    
    # Run inference
    predictions = model([image_tensor])
    pred = predictions[0]
    
    # Filter by confidence threshold
    keep_indices = pred["scores"] >= confidence_threshold
    
    filtered_predictions = {
        "boxes": pred["boxes"][keep_indices].cpu().numpy(),
        "labels": pred["labels"][keep_indices].cpu().numpy(),
        "scores": pred["scores"][keep_indices].cpu().numpy()
    }
    
    return original_image, filtered_predictions


def visualize_predictions_matplotlib(
    image: np.ndarray,
    predictions: Dict,
    save_path: Optional[str] = None,
    figsize: Tuple[int, int] = (12, 10)
):
    """
    Visualize predictions using matplotlib.
    
    Args:
        image: Original image as numpy array (H, W, C)
        predictions: Dictionary with boxes, labels, scores
        save_path: Optional path to save the visualization
        figsize: Figure size
    """
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    
    boxes = predictions["boxes"]
    labels = predictions["labels"]
    scores = predictions["scores"]
    
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = box
        width = x2 - x1
        height = y2 - y1
        
        # Get class name
        class_name = IDX_TO_CLASS.get(label, f"Class {label}")
        
        # Get color for this class
        color = COLORS[label % len(COLORS)]
        
        # Draw bounding box
        rect = patches.Rectangle(
            (x1, y1), width, height,
            linewidth=2,
            edgecolor=color,
            facecolor='none'
        )
        ax.add_patch(rect)
        
        # Draw label with background
        label_text = f"{class_name}: {score:.2f}"
        ax.text(
            x1, y1 - 5,
            label_text,
            fontsize=12,
            fontweight='bold',
            color='white',
            bbox=dict(
                boxstyle='round,pad=0.3',
                facecolor=color,
                edgecolor=color,
                alpha=0.8
            )
        )
    
    ax.axis('off')
    ax.set_title(f"Sign Language Detection - {len(boxes)} detections", fontsize=14)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved visualization to {save_path}")
    
    plt.show()


def visualize_predictions_cv2(
    image: np.ndarray,
    predictions: Dict,
    save_path: Optional[str] = None
) -> np.ndarray:
    """
    Visualize predictions using OpenCV.
    
    Args:
        image: Original image as numpy array (H, W, C) in RGB format
        predictions: Dictionary with boxes, labels, scores
        save_path: Optional path to save the visualization
        
    Returns:
        Annotated image as numpy array
    """
    # Convert RGB to BGR for OpenCV
    annotated_image = cv2.cvtColor(image.copy(), cv2.COLOR_RGB2BGR)
    
    boxes = predictions["boxes"]
    labels = predictions["labels"]
    scores = predictions["scores"]
    
    for box, label, score in zip(boxes, labels, scores):
        x1, y1, x2, y2 = map(int, box)
        
        # Get class name
        class_name = IDX_TO_CLASS.get(label, f"Class {label}")
        
        # Get color (BGR format for OpenCV)
        color = tuple(int(c * 255) for c in COLORS[label % len(COLORS)][:3])
        color = (color[2], color[1], color[0])  # RGB to BGR
        
        # Draw bounding box
        cv2.rectangle(annotated_image, (x1, y1), (x2, y2), color, 2)
        
        # Draw label background
        label_text = f"{class_name}: {score:.2f}"
        (text_width, text_height), baseline = cv2.getTextSize(
            label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
        )
        cv2.rectangle(
            annotated_image,
            (x1, y1 - text_height - 10),
            (x1 + text_width + 5, y1),
            color,
            -1
        )
        
        # Draw label text
        cv2.putText(
            annotated_image,
            label_text,
            (x1 + 2, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 255, 255),
            2
        )
    
    if save_path:
        cv2.imwrite(save_path, annotated_image)
        print(f"Saved visualization to {save_path}")
    
    # Convert back to RGB for display
    return cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)


def run_inference_demo(
    model: nn.Module,
    image_path: str,
    device: torch.device,
    confidence_threshold: float = 0.5
):
    """
    Complete inference demo: load image, predict, and visualize.
    
    Args:
        model: Trained Faster R-CNN model
        image_path: Path to test image
        device: Device to run on
        confidence_threshold: Minimum confidence threshold
    """
    print(f"Running inference on: {image_path}")
    print(f"Confidence threshold: {confidence_threshold}")
    
    # Run prediction
    image, predictions = predict_single_image(
        model, image_path, device, confidence_threshold
    )
    
    # Print detected signs
    num_detections = len(predictions["boxes"])
    print(f"\nDetected {num_detections} signs:")
    for i, (label, score) in enumerate(zip(predictions["labels"], predictions["scores"])):
        class_name = IDX_TO_CLASS.get(label, f"Class {label}")
        print(f"  {i+1}. {class_name} (confidence: {score:.3f})")
    
    # Visualize with matplotlib
    visualize_predictions_matplotlib(image, predictions)


# ==========================================
# Run inference demo (uncomment with actual image)
# ==========================================

# # Load trained model
# checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pth"))
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()

# # Run demo
# run_inference_demo(
#     model=model,
#     image_path="./test_images/test_sign.jpg",
#     device=DEVICE,
#     confidence_threshold=0.5
# )

print("Inference and visualization functions defined!")

## 9. Feature Map Extraction using Forward Hooks

This section demonstrates how to use PyTorch's `register_forward_hook` mechanism to capture intermediate feature maps during the forward pass.

**How forward hooks work:**
1. Register a hook function on a target layer
2. During forward pass, the hook captures the layer's input and output
3. Store the captured features for later visualization

**Target layers:**
- **Backbone layers (layer1-layer4):** Extract hierarchical features from the ResNet backbone
- **FPN outputs (p2-p5):** Multi-scale feature maps from Feature Pyramid Network
- **RPN features:** Region Proposal Network intermediate outputs

In [ ]:
class FeatureMapExtractor:
    """
    Extract feature maps from intermediate layers using forward hooks.
    
    Forward hooks allow us to intercept the forward pass and capture
    the input/output tensors at any layer without modifying the model.
    
    Usage:
        extractor = FeatureMapExtractor(model)
        extractor.register_hooks(['backbone.body.layer1', 'backbone.body.layer2'])
        predictions = extractor.extract(image_tensor)
        feature_maps = extractor.get_feature_maps()
    """
    
    def __init__(self, model: nn.Module):
        """
        Initialize the feature map extractor.
        
        Args:
            model: The PyTorch model to extract features from
        """
        self.model = model
        self.feature_maps = {}  # Store extracted feature maps
        self.hooks = []  # Store hook handles for cleanup
        
    def _get_hook_fn(self, name: str):
        """
        Create a hook function that stores the output of a layer.
        
        The hook function signature is: hook(module, input, output)
        - module: The layer being hooked
        - input: Tuple of inputs to the layer
        - output: The layer's output tensor
        
        Args:
            name: Name to identify this layer's features
            
        Returns:
            Hook function
        """
        def hook_fn(module, input, output):
            # Handle different output types
            if isinstance(output, torch.Tensor):
                # Store detached copy to avoid memory leaks
                self.feature_maps[name] = output.detach().cpu()
            elif isinstance(output, dict):
                # FPN returns dict of feature maps
                self.feature_maps[name] = {
                    k: v.detach().cpu() for k, v in output.items()
                }
            elif isinstance(output, (list, tuple)):
                # Some layers return lists/tuples
                self.feature_maps[name] = [
                    o.detach().cpu() if isinstance(o, torch.Tensor) else o 
                    for o in output
                ]
        return hook_fn
    
    def _get_layer_by_name(self, name: str) -> nn.Module:
        """
        Get a layer by its dot-separated name path.
        
        Example: 'backbone.body.layer1' -> model.backbone.body.layer1
        
        Args:
            name: Dot-separated path to the layer
            
        Returns:
            The target layer module
        """
        parts = name.split('.')
        layer = self.model
        for part in parts:
            layer = getattr(layer, part)
        return layer
    
    def register_hooks(self, layer_names: List[str]):
        """
        Register forward hooks on specified layers.
        
        Args:
            layer_names: List of layer names to hook
        """
        self.clear_hooks()  # Remove any existing hooks
        
        for name in layer_names:
            try:
                layer = self._get_layer_by_name(name)
                hook = layer.register_forward_hook(self._get_hook_fn(name))
                self.hooks.append(hook)
                print(f"  ✓ Registered hook on: {name}")
            except AttributeError as e:
                print(f"  ✗ Could not find layer: {name}")
    
    def clear_hooks(self):
        """Remove all registered hooks."""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
        self.feature_maps = {}
    
    @torch.no_grad()
    def extract(
        self, 
        image_tensor: torch.Tensor,
        device: torch.device = DEVICE
    ) -> List[Dict]:
        """
        Run forward pass and extract feature maps.
        
        Args:
            image_tensor: Input image tensor (C, H, W)
            device: Device to run on
            
        Returns:
            Model predictions
        """
        self.model.eval()
        self.feature_maps = {}  # Clear previous features
        
        # Ensure correct input format
        if image_tensor.dim() == 3:
            image_tensor = image_tensor.unsqueeze(0)
        
        image_tensor = image_tensor.to(device)
        
        # Run forward pass (hooks will capture features)
        predictions = self.model(image_tensor)
        
        return predictions
    
    def get_feature_maps(self) -> Dict[str, torch.Tensor]:
        """Get all captured feature maps."""
        return self.feature_maps
    
    def __del__(self):
        """Cleanup hooks when object is destroyed."""
        self.clear_hooks()


def get_backbone_layer_names(model_type: str = "resnet50") -> List[str]:
    """
    Get the layer names for hooking based on model type.
    
    Args:
        model_type: Either "resnet50" or "mobilenet_v3"
        
    Returns:
        List of layer names to hook
    """
    if model_type == "resnet50":
        # ResNet-50 FPN backbone structure
        return [
            "backbone.body.conv1",   # Initial conv
            "backbone.body.layer1",  # 256 channels, stride 4
            "backbone.body.layer2",  # 512 channels, stride 8
            "backbone.body.layer3",  # 1024 channels, stride 16
            "backbone.body.layer4",  # 2048 channels, stride 32
            "backbone.fpn",          # Feature Pyramid Network
        ]
    else:
        # MobileNetV3 has different structure
        return [
            "backbone.body.0",    # Initial layers
            "backbone.body.4",    # Mid layers
            "backbone.body.8",    # Mid layers
            "backbone.body.12",   # Late layers
            "backbone.fpn",       # Feature Pyramid Network
        ]


def get_rpn_layer_names() -> List[str]:
    """Get RPN layer names for hooking."""
    return [
        "rpn.head.conv",      # RPN conv layer
        "rpn.head.cls_logits", # Objectness scores
        "rpn.head.bbox_pred",  # Box deltas
    ]


# Print available layers for reference
print("Model structure for hooking:")
print("-" * 50)

# Show backbone structure
print("\nBackbone layers (ResNet-50 FPN):")
for name, module in model.backbone.named_modules():
    if name and '.' not in name:  # Top-level submodules
        print(f"  backbone.{name}: {type(module).__name__}")

print("\nRPN layers:")
for name, module in model.rpn.named_modules():
    if name and name.count('.') <= 1:  # First two levels
        print(f"  rpn.{name}: {type(module).__name__}")

## 10. Backbone Feature Map Visualization

Visualize feature maps from the backbone network (ResNet-50) at different stages. Each stage learns increasingly abstract representations:
- **Layer 1-2:** Low-level features (edges, textures)
- **Layer 3-4:** High-level features (shapes, patterns)

In [ ]:
def visualize_feature_maps(
    feature_map: torch.Tensor,
    layer_name: str,
    num_channels: int = 16,
    figsize: Tuple[int, int] = (16, 8),
    save_path: Optional[str] = None
):
    """
    Visualize multiple channels from a feature map.
    
    Args:
        feature_map: Feature tensor of shape (B, C, H, W)
        layer_name: Name of the layer for the title
        num_channels: Number of channels to visualize
        figsize: Figure size
        save_path: Optional path to save the figure
    """
    # Handle batch dimension
    if feature_map.dim() == 4:
        feature_map = feature_map[0]  # Take first image in batch
    
    # Get number of channels to display
    num_channels = min(num_channels, feature_map.shape[0])
    
    # Calculate grid dimensions
    cols = 8
    rows = (num_channels + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten() if rows > 1 else [axes] if rows == 1 and cols == 1 else axes.flatten()
    
    for i in range(num_channels):
        channel = feature_map[i].numpy()
        
        # Normalize for visualization
        channel = (channel - channel.min()) / (channel.max() - channel.min() + 1e-8)
        
        axes[i].imshow(channel, cmap='viridis')
        axes[i].axis('off')
        axes[i].set_title(f'Ch {i}', fontsize=8)
    
    # Hide empty subplots
    for i in range(num_channels, len(axes)):
        axes[i].axis('off')
    
    fig.suptitle(f'Feature Maps: {layer_name}\nShape: {tuple(feature_map.shape)}', fontsize=12)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()


def visualize_backbone_features(
    model: nn.Module,
    image_path: str,
    device: torch.device = DEVICE
):
    """
    Extract and visualize backbone feature maps.
    
    Args:
        model: Faster R-CNN model
        image_path: Path to the input image
        device: Device to run on
    """
    print("=" * 60)
    print("BACKBONE FEATURE MAP VISUALIZATION")
    print("=" * 60)
    
    # Load and preprocess image
    image = Image.open(image_path).convert("RGB")
    original_image = np.array(image)
    
    image_tensor = T.functional.to_image(image)
    image_tensor = T.functional.to_dtype(image_tensor, dtype=torch.float32, scale=True)
    
    # Create extractor and register hooks
    extractor = FeatureMapExtractor(model)
    
    backbone_layers = [
        "backbone.body.layer1",
        "backbone.body.layer2", 
        "backbone.body.layer3",
        "backbone.body.layer4",
    ]
    
    print("\nRegistering hooks on backbone layers...")
    extractor.register_hooks(backbone_layers)
    
    # Run forward pass
    print("\nRunning forward pass...")
    predictions = extractor.extract(image_tensor, device)
    
    # Get captured features
    feature_maps = extractor.get_feature_maps()
    
    # Display original image
    print("\n--- Original Image ---")
    plt.figure(figsize=(8, 6))
    plt.imshow(original_image)
    plt.axis('off')
    plt.title("Input Image")
    plt.show()
    
    # Visualize each layer's features
    print("\n--- Feature Maps by Layer ---")
    for layer_name in backbone_layers:
        if layer_name in feature_maps:
            fm = feature_maps[layer_name]
            print(f"\n{layer_name}: shape = {tuple(fm.shape)}")
            visualize_feature_maps(fm, layer_name, num_channels=16)
    
    # Cleanup hooks
    extractor.clear_hooks()
    print("\nHooks cleared. Backbone visualization complete!")
    
    return feature_maps


# ==========================================
# Run backbone visualization (uncomment with actual image)
# ==========================================

# backbone_features = visualize_backbone_features(
#     model=model,
#     image_path="./test_images/test_sign.jpg",
#     device=DEVICE
# )

print("Backbone visualization function defined!")

## 11. RPN and FPN Feature Map Visualization

Visualize the Feature Pyramid Network (FPN) outputs and Region Proposal Network (RPN) features:
- **FPN outputs (P2-P5):** Multi-scale feature maps at different resolutions
- **RPN features:** Objectness scores showing where the network expects objects

In [ ]:
def visualize_fpn_features(
    fpn_output: Dict[str, torch.Tensor],
    original_image: np.ndarray,
    figsize: Tuple[int, int] = (16, 10),
    save_path: Optional[str] = None
):
    """
    Visualize Feature Pyramid Network (FPN) multi-scale outputs.
    
    FPN produces feature maps at different scales:
    - P2: Highest resolution, lowest-level features (1/4 of input)
    - P3: 1/8 of input resolution
    - P4: 1/16 of input resolution
    - P5: Lowest resolution, highest-level features (1/32 of input)
    - pool: Global pooled features
    
    Args:
        fpn_output: Dictionary of FPN outputs {level_name: tensor}
        original_image: Original input image for reference
        figsize: Figure size
        save_path: Optional path to save
    """
    # Filter to only tensor outputs
    fpn_levels = {k: v for k, v in fpn_output.items() if isinstance(v, torch.Tensor)}
    num_levels = len(fpn_levels)
    
    fig, axes = plt.subplots(2, max(num_levels, 3), figsize=figsize)
    
    # Show original image
    axes[0, 0].imshow(original_image)
    axes[0, 0].set_title("Original Image")
    axes[0, 0].axis('off')
    
    # Plot each FPN level
    for i, (level_name, feature_map) in enumerate(fpn_levels.items()):
        if i >= axes.shape[1]:
            break
            
        # Take first channel mean for visualization
        if feature_map.dim() == 4:
            fm_vis = feature_map[0].mean(dim=0).numpy()
        else:
            fm_vis = feature_map.mean(dim=0).numpy() if feature_map.dim() == 3 else feature_map.numpy()
        
        # Normalize
        fm_vis = (fm_vis - fm_vis.min()) / (fm_vis.max() - fm_vis.min() + 1e-8)
        
        ax = axes[0, i + 1] if i < axes.shape[1] - 1 else axes[1, i - axes.shape[1] + 1]
        ax.imshow(fm_vis, cmap='hot')
        ax.set_title(f"{level_name}\n{feature_map.shape[-2]}x{feature_map.shape[-1]}")
        ax.axis('off')
    
    # Show channel diversity for first FPN level
    ax_row2 = axes[1]
    first_level = list(fpn_levels.values())[0]
    if first_level.dim() == 4:
        first_level = first_level[0]
    
    num_show = min(6, first_level.shape[0])
    for i in range(num_show):
        if i < len(ax_row2):
            channel = first_level[i].numpy()
            channel = (channel - channel.min()) / (channel.max() - channel.min() + 1e-8)
            ax_row2[i].imshow(channel, cmap='viridis')
            ax_row2[i].set_title(f"Ch {i}", fontsize=8)
            ax_row2[i].axis('off')
    
    # Hide unused axes
    for ax in axes.flatten():
        if not ax.has_data():
            ax.axis('off')
    
    fig.suptitle("Feature Pyramid Network (FPN) Multi-Scale Features", fontsize=14)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved to {save_path}")
    
    plt.show()


def visualize_rpn_features(
    model: nn.Module,
    image_path: str,
    device: torch.device = DEVICE
):
    """
    Extract and visualize RPN (Region Proposal Network) features.
    
    The RPN predicts:
    - Objectness scores: Probability that each anchor contains an object
    - Box deltas: Adjustments to anchor boxes to better fit objects
    
    Args:
        model: Faster R-CNN model
        image_path: Path to input image
        device: Device to run on
    """
    print("=" * 60)
    print("RPN AND FPN FEATURE MAP VISUALIZATION")
    print("=" * 60)
    
    # Load image
    image = Image.open(image_path).convert("RGB")
    original_image = np.array(image)
    
    image_tensor = T.functional.to_image(image)
    image_tensor = T.functional.to_dtype(image_tensor, dtype=torch.float32, scale=True)
    
    # Create extractor
    extractor = FeatureMapExtractor(model)
    
    # Register hooks for FPN and RPN
    hook_layers = [
        "backbone.fpn",       # FPN output
        "rpn.head.conv",      # RPN conv features
    ]
    
    print("\nRegistering hooks on FPN and RPN layers...")
    extractor.register_hooks(hook_layers)
    
    # Run forward pass
    print("\nRunning forward pass...")
    predictions = extractor.extract(image_tensor, device)
    
    feature_maps = extractor.get_feature_maps()
    
    # Visualize FPN features
    print("\n--- FPN Multi-Scale Features ---")
    if "backbone.fpn" in feature_maps:
        fpn_features = feature_maps["backbone.fpn"]
        if isinstance(fpn_features, dict):
            visualize_fpn_features(fpn_features, original_image)
    
    # Visualize RPN conv features
    print("\n--- RPN Head Conv Features ---")
    if "rpn.head.conv" in feature_maps:
        rpn_conv = feature_maps["rpn.head.conv"]
        if isinstance(rpn_conv, list):
            # RPN head is applied to each FPN level
            fig, axes = plt.subplots(1, min(len(rpn_conv), 5), figsize=(16, 4))
            if len(rpn_conv) == 1:
                axes = [axes]
            
            for i, (rpn_fm, ax) in enumerate(zip(rpn_conv, axes)):
                if rpn_fm.dim() == 4:
                    rpn_fm = rpn_fm[0]
                # Show mean activation
                vis = rpn_fm.mean(dim=0).numpy()
                vis = (vis - vis.min()) / (vis.max() - vis.min() + 1e-8)
                ax.imshow(vis, cmap='hot')
                ax.set_title(f"RPN Level {i}\n{rpn_fm.shape[-2]}x{rpn_fm.shape[-1]}")
                ax.axis('off')
            
            plt.suptitle("RPN Conv Features (Mean Channel Activation)", fontsize=12)
            plt.tight_layout()
            plt.show()
        else:
            visualize_feature_maps(rpn_conv, "RPN Conv Output", num_channels=16)
    
    # Show predictions overlaid on image
    print("\n--- Region Proposals on Image ---")
    pred = predictions[0]
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    ax.imshow(original_image)
    
    # Draw top predictions
    boxes = pred["boxes"].cpu().numpy()[:10]  # Top 10
    scores = pred["scores"].cpu().numpy()[:10]
    labels = pred["labels"].cpu().numpy()[:10]
    
    for box, score, label in zip(boxes, scores, labels):
        if score < 0.3:  # Skip low confidence
            continue
        x1, y1, x2, y2 = box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2,
            edgecolor='lime',
            facecolor='none'
        )
        ax.add_patch(rect)
        class_name = IDX_TO_CLASS.get(label, f"Class {label}")
        ax.text(x1, y1-5, f"{class_name}: {score:.2f}", 
                color='white', fontsize=10,
                bbox=dict(boxstyle='round', facecolor='green', alpha=0.7))
    
    ax.axis('off')
    ax.set_title("Top 10 Detections")
    plt.tight_layout()
    plt.show()
    
    # Cleanup
    extractor.clear_hooks()
    print("\nRPN/FPN visualization complete!")


# ==========================================
# Run RPN visualization (uncomment with actual image)
# ==========================================

# visualize_rpn_features(
#     model=model,
#     image_path="./test_images/test_sign.jpg",
#     device=DEVICE
# )

print("RPN/FPN visualization function defined!")

## 12. Save Model for Deployment

Save the trained model in a format suitable for deployment with FastAPI.

In [ ]:
def save_model_for_deployment(
    model: nn.Module,
    save_path: str = "./model_deployment.pth",
    include_config: bool = True
):
    """
    Save the model for deployment with FastAPI.
    
    Args:
        model: Trained Faster R-CNN model
        save_path: Path to save the model
        include_config: Whether to include configuration metadata
    """
    # Prepare save dictionary
    save_dict = {
        "model_state_dict": model.state_dict(),
    }
    
    if include_config:
        save_dict.update({
            "num_classes": NUM_CLASSES,
            "class_to_idx": CLASS_TO_IDX,
            "idx_to_class": IDX_TO_CLASS,
            "model_config": {
                "backbone": "resnet50",
                "pretrained_backbone": True,
            }
        })
    
    torch.save(save_dict, save_path)
    print(f"Model saved to: {save_path}")
    print(f"File size: {os.path.getsize(save_path) / (1024*1024):.2f} MB")


def load_model_for_deployment(
    model_path: str = "./model_deployment.pth",
    device: torch.device = DEVICE
) -> nn.Module:
    """
    Load a saved model for deployment.
    
    Args:
        model_path: Path to the saved model
        device: Device to load the model on
        
    Returns:
        Loaded and configured model in eval mode
    """
    # Load checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    
    # Get configuration
    num_classes = checkpoint.get("num_classes", NUM_CLASSES)
    
    # Create model with same configuration
    model = create_faster_rcnn_model(
        num_classes=num_classes,
        backbone="resnet50",
        pretrained=False  # We're loading our own weights
    )
    
    # Load state dict
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()
    
    print(f"Model loaded from: {model_path}")
    print(f"Number of classes: {num_classes}")
    
    return model


# ==========================================
# Save model (uncomment after training)
# ==========================================

# save_model_for_deployment(model, "./model_deployment.pth")

# # Verify loading works
# loaded_model = load_model_for_deployment("./model_deployment.pth")

print("Model saving functions defined!")
print("\n" + "=" * 60)
print("NOTEBOOK COMPLETE")
print("=" * 60)
print("""
Next steps:
1. Update IMAGES_DIR and ANNOTATIONS_DIR with your data paths
2. Run create_data_loaders() to prepare your dataset
3. Run train_model() to train the model
4. Run evaluate_map() to compute mAP metrics
5. Save the model with save_model_for_deployment()
6. Use the saved model in app.py for real-time inference
""")